# Importing Libraries

- [Dataset](#CorrelationInsights)

## Before you run the notebook

Place `inventory.db` (containing the `vendor_sales_summary` table) or `vendor_sales_summary.csv` in the notebook folder. A CSV inside `data/` is also supported. The notebook cannot calculate results until the source data is available.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sqlite3
from scipy.stats import ttest_ind
import scipy.stats as stats
warnings.filterwarnings('ignore')

# Dataset

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("inventory.db")
CSV_CANDIDATES = [
    Path("vendor_sales_summary.csv"),
    Path("data/vendor_sales_summary.csv"),
]

if DB_PATH.exists():
    conn = sqlite3.connect(DB_PATH)
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='vendor_sales_summary'",
        conn
    )
    if not tables.empty:
        df = pd.read_sql_query("SELECT * FROM vendor_sales_summary", conn)
    else:
        conn.close()
        csv_path = next((p for p in CSV_CANDIDATES if p.exists()), None)
        if csv_path is None:
            raise FileNotFoundError(
                "The database exists, but table 'vendor_sales_summary' was not found. "
                "Place vendor_sales_summary.csv in the notebook folder or data/ folder."
            )
        df = pd.read_csv(csv_path)
else:
    csv_path = next((p for p in CSV_CANDIDATES if p.exists()), None)
    if csv_path is None:
        raise FileNotFoundError(
            "Could not find inventory.db or vendor_sales_summary.csv. "
            "Place inventory.db (with table 'vendor_sales_summary') or the CSV "
            "in the notebook folder or data/ folder, then rerun this cell."
        )
    df = pd.read_csv(csv_path)

print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

In [ ]:
df.to_csv("vendor_sales_summary.csv", index=False)
print("Saved current source data to vendor_sales_summary.csv")

# Exploratory Data Analysis 
- Previously, we examined the various tables in the database to identify key variables, understand their relationships, and determine which ones should be included in the final analysis.

- In this phase of EDA, we will analyze the resultant table to gain insights into the distribution of each column. This will help us understand data patterns, identify anomalies, and ensure data quality before proceeding with further analysis.


In [ ]:
# Summary statistics for numerical columns
summary_stats = df.describe().T  
display(summary_stats)

In [ ]:
# Mode for each numerical column
mode_values = df.mode().iloc[0]
print("\nMode Values:\n\n", mode_values)


In [ ]:
import math

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
if not numerical_cols:
    print("No numeric columns available for distribution plots.")
else:
    ncols = 4
    nrows = math.ceil(len(numerical_cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, col in zip(axes, numerical_cols):
        sns.histplot(df[col].dropna(), kde=True, bins=30, ax=ax)
        ax.set_title(col)

    for ax in axes[len(numerical_cols):]:
        ax.remove()

    plt.tight_layout()
    plt.show()

In [ ]:
import math

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
if not numerical_cols:
    print("No numeric columns available for boxplots.")
else:
    ncols = 4
    nrows = math.ceil(len(numerical_cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, col in zip(axes, numerical_cols):
        sns.boxplot(y=df[col].dropna(), ax=ax)
        ax.set_title(col)

    for ax in axes[len(numerical_cols):]:
        ax.remove()

    plt.tight_layout()
    plt.show()

## Summary Statistics Insights:
**Negative & Zero Values:**

- Gross Profit: Minimum value is -52,002.78, indicating losses. Some products or transactions may be selling at a loss due to high costs or selling at discounts lower than the purchase price..
- Profit Margin: Has a minimum of -∞, which suggests cases where revenue is zero or even lower than costs.
- Total Sales Quantity & Sales Dollars: Minimum values are 0, meaning some products were purchased but never sold. These could be slow-moving or obsolete stock.

**Outliers Indicated by High Standard Deviations:**

- Purchase & Actual Prices: The max values (5,681.81 & 7,499.99) are significantly higher than the mean (24.39 & 35.64), indicating potential premium products.
- Freight Cost: Huge variation, from 0.09 to 257,032.07, suggests logistics inefficiencies or bulk shipments.
- Stock Turnover: Ranges from 0 to 274.5, implying some products sell extremely fast while others remain in stock indefinitely. Value more than 1 indicates that Sold quantity for that product is higher than purchased quantity due to either sales are being fulfilled from older stock.

In [ ]:
# let's filter the data by removing inconsistencies
df = pd.read_sql_query("""SELECT * 
FROM vendor_sales_summary
WHERE GrossProfit > 0
AND ProfitMargin > 0
AND TotalSalesQuantity > 0""",conn)

In [ ]:
df.shape

In [ ]:
import math

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
if not numerical_cols:
    print("No numeric columns available for distribution plots.")
else:
    ncols = 4
    nrows = math.ceil(len(numerical_cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, col in zip(axes, numerical_cols):
        sns.histplot(df[col].dropna(), kde=True, bins=30, ax=ax)
        ax.set_title(col)

    for ax in axes[len(numerical_cols):]:
        ax.remove()

    plt.tight_layout()
    plt.show()

In [ ]:
# Count Plots for Categorical Columns
categorical_cols = ["VendorName", "Description"]

plt.figure(figsize=(12, 5))
for i, col in enumerate(categorical_cols):
    plt.subplot(1, 2, i+1)
    sns.countplot(y=df[col], order=df[col].value_counts().index[:10])  # Top 10 categories
    plt.title(f"Count Plot of {col}")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))
correlation_matrix = df[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

## Correlation Insights
- PurchasePrice has weak correlations with TotalSalesDollars (-0.012) and GrossProfit (-0.016), suggesting that price variations do not significantly impact sales revenue or profit.
- Strong correlation between total purchase quantity and total sales quantity (0.999), confirming efficient inventory turnover.
- Negative correlation between profit margin & total sales price (-0.179) suggests that as sales price increases, margins decrease, possibly due to competitive pricing pressures.
- StockTurnover has weak negative correlations with both GrossProfit (-0.038) and ProfitMargin (-0.055), indicating that faster turnover does not necessarily result in higher profitability.


# Data Analysis

#### Identify Brands that needs Promotional or Pricing Adjustments which exhibit lower sales performance but higher profit margins.

In [ ]:
brand_performance = df.groupby('Description').agg({
    'TotalSalesDollars': 'sum',  # Sales performance metric
    'ProfitMargin': 'mean'       # Average profit margin
}).reset_index()

brand_performance.sort_values('ProfitMargin')


In [ ]:
# threshold for "low sales" (bottom 15%) and "high margin" (top 15%)
low_sales_threshold = brand_performance['TotalSalesDollars'].quantile(0.15)
high_margin_threshold = brand_performance['ProfitMargin'].quantile(0.85)

# Filter brands with low sales but high profit margins
target_brands = brand_performance[
    (brand_performance['TotalSalesDollars'] <= low_sales_threshold) &
    (brand_performance['ProfitMargin'] >= high_margin_threshold)
]
print("Brands with Low Sales but High Profit Margins:")
display(target_brands.sort_values('TotalSalesDollars'))

In [ ]:
brand_performance = brand_performance[brand_performance['TotalSalesDollars']<10000] # for better visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=brand_performance, x='TotalSalesDollars', y='ProfitMargin', color="blue", label="All Brands", alpha = 0.2)
sns.scatterplot(data=target_brands, x='TotalSalesDollars', y='ProfitMargin', color="red", label="Target Brands")

plt.axhline(high_margin_threshold, linestyle='--', color='black', label="High Margin Threshold")
plt.axvline(low_sales_threshold, linestyle='--', color='black', label="Low Sales Threshold")

plt.xlabel("Total Sales ($)")
plt.ylabel("Profit Margin (%)")
plt.title("Brands for Promotional or Pricing Adjustments")
plt.legend()
plt.grid(True)
plt.show()

#### Which vendors and brands demonstrate the highest sales performance?

In [ ]:
def format_dollars(value):
    if value >= 1_000_000:
        return f"{value / 1_000_000:.2f}M"
    elif value >= 1_000:
        return f"{value / 1_000:.2f}K"
    else:
        return str(value)

In [ ]:
# Top Vendors & Brands by Sales Performance
top_vendors = df.groupby("VendorName")["TotalSalesDollars"].sum().nlargest(10)
top_brands = df.groupby("Description")["TotalSalesDollars"].sum().nlargest(10)
top_vendors

In [ ]:
top_vendors.apply(lambda x:format_dollars(x))

In [ ]:
plt.figure(figsize=(15, 5))

# Plot for Top Vendors
plt.subplot(1, 2, 1)
ax1 = sns.barplot(y=top_vendors.index, x=top_vendors.values, palette="Blues_r")
plt.title("Top 10 Vendors by Sales")

for bar in ax1.patches:
    ax1.text(bar.get_width() + (bar.get_width() * 0.02), 
             bar.get_y() + bar.get_height() / 2, 
             format_dollars(bar.get_width()), 
             ha='left', va='center', fontsize=10, color='black')

# Plot for Top Brands
plt.subplot(1, 2, 2)
ax2 = sns.barplot(y=top_brands.index.astype(str), x=top_brands.values, palette="Reds_r")
plt.title("Top 10 Brands by Sales")

for bar in ax2.patches:
    ax2.text(bar.get_width() + (bar.get_width() * 0.02), 
             bar.get_y() + bar.get_height() / 2, 
             format_dollars(bar.get_width()), 
             ha='left', va='center', fontsize=10, color='black')

plt.tight_layout()
plt.show()

### Which vendors contribute the most to total purchase dollars?

In [ ]:
# Rank vendors by total purchase dollars
vendor_performance = df.groupby("VendorName", as_index=False).agg({
    "TotalPurchaseDollars": "sum",
    "GrossProfit": "sum",
    "TotalSalesDollars": "sum"
})

total_purchase_dollars = vendor_performance["TotalPurchaseDollars"].sum()
if total_purchase_dollars:
    vendor_performance["Purchase_Contribution%"] = (
        vendor_performance["TotalPurchaseDollars"] / total_purchase_dollars * 100
    )
else:
    vendor_performance["Purchase_Contribution%"] = 0.0

vendor_performance = vendor_performance.sort_values(
    by="TotalPurchaseDollars", ascending=False
).round(2)

# Work on a display copy so numeric values remain numeric for later charts.
top_vendors = vendor_performance.head(10).copy()
display_vendors = top_vendors.copy()
for col in ["TotalSalesDollars", "TotalPurchaseDollars", "GrossProfit"]:
    display_vendors[col] = display_vendors[col].apply(format_dollars)

display_vendors

In [ ]:
top_vendors['Cumulative_Contribution%'] = top_vendors['Purchase_Contribution%'].cumsum()

fig, ax1 = plt.subplots(figsize=(10, 6))

# Bar plot for Purchase Contribution%
sns.barplot(x=top_vendors['VendorName'], y=top_vendors['Purchase_Contribution%'],  palette="mako", ax=ax1)

for i, value in enumerate(top_vendors['Purchase_Contribution%']):
    ax1.text(i, value - 1, str(value)+'%', ha='center', fontsize=10, color='white')

# Line Plot for Cumulative Contribution%
ax2 = ax1.twinx()
ax2.plot(top_vendors['VendorName'], top_vendors['Cumulative_Contribution%'], color='red', marker='o', linestyle='dashed', label='Cumulative %')

ax1.set_xticklabels(top_vendors['VendorName'], rotation=90)
ax1.set_ylabel('Purchase Contribution %', color='blue')
ax2.set_ylabel('Cumulative Contribution %', color='red')
ax1.set_xlabel('Vendors')
ax1.set_title('Pareto Chart: Vendor Contribution to Total Purchases')

ax2.axhline(y=100, color='gray', linestyle='dashed', alpha=0.7)
ax2.legend(loc='upper right')

plt.show()

### How much of total procurement is dependent on the top vendors?

In [ ]:
print(f"Total Purchase Contribution of top 10 vendors is {round(top_vendors['Purchase_Contribution%'].sum(),2)} %")

vendors = list(top_vendors['VendorName'].values)
purchase_contributions = list(top_vendors['Purchase_Contribution%'].values)
total_contribution = sum(purchase_contributions)
remaining_contribution = 100 - total_contribution

# Append "Other Vendors" category
vendors.append("Other Vendors")
purchase_contributions.append(remaining_contribution)

# Donut Chart
fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(purchase_contributions, labels=vendors, autopct='%1.1f%%',
                                  startangle=140, pctdistance=0.85, colors=plt.cm.Paired.colors)

# Draw a white circle in the center to create a "donut" effect
centre_circle = plt.Circle((0, 0), 0.70, fc='white')
fig.gca().add_artist(centre_circle)

# Add Total Contribution annotation in the center
plt.text(0, 0, f"Top 10 Total:\n{total_contribution:.2f}%", fontsize=14, fontweight='bold', ha='center', va='center')

plt.title("Top 10 Vendor's Purchase Contribution (%)")
plt.show()

The remaining vendors contribute only 34.31%, meaning they are not utilized effectively or may not be as competitive.
If vendor dependency is too high, consider identifying new suppliers to reduce risk.

#### Does purchasing in bulk reduce the unit price, and what is the optimal purchase volume for cost savings?

In [ ]:
# Calculate Unit Purchase Price per Vendor & Volume Group
df["UnitPurchasePrice"] = df["TotalPurchaseDollars"] / df["TotalPurchaseQuantity"]

# Group by Order Sizes (e.g., Small, Medium, Large Purchases)
df["OrderSize"] = pd.qcut(df["TotalPurchaseQuantity"], q=3, labels=["Small", "Medium", "Large"])

# Analyze Cost Savings per Order Size
bulk_purchase_analysis = df.groupby("OrderSize")["UnitPurchasePrice"].mean().reset_index()
print(bulk_purchase_analysis)


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="OrderSize", y="UnitPurchasePrice", palette="Set2")
plt.title("Impact of Bulk Purchasing on Unit Price")
plt.xlabel("Order Size")
plt.ylabel("Average Unit Purchase Price")
plt.show()


- Vendors buying in bulk (Large Order Size) get the lowest unit price ($10.78 per unit), meaning higher margins if they can manage inventory efficiently.
- The price difference between Small and Large orders is substantial (~72% reduction in unit cost)
- This suggests that bulk pricing strategies successfully encourage vendors to purchase in larger volumes, leading to higher overall sales despite lower per-unit revenue.

#### Which vendors have low inventory turnover, indicating excess stock and slow-moving products?

In [ ]:


# Identify Low Inventory Turnover Vendors
low_turnover_vendors = df[df["StockTurnover"] < 1].groupby("VendorName")["StockTurnover"].mean().reset_index()

# Sort by Lowest Turnover
low_turnover_vendors = low_turnover_vendors.sort_values(by="StockTurnover", ascending=True)
low_turnover_vendors.head(10)


- Slow-moving inventory increases holding costs (warehouse rent, insurance, depreciation)
- Identifying vendors with low inventory turnover is critical for business efficiency, cost reduction, and profitability


#### How much capital is locked in unsold inventory per vendor, and which vendors contribute the most to it?

In [ ]:
# Calculate Unsold Inventory Value
df["UnsoldInventoryValue"] = (df["TotalPurchaseQuantity"] - df["TotalSalesQuantity"]) * df["PurchasePrice"]
print('Total Unsold Capital:', format_dollars(df["UnsoldInventoryValue"].sum()))

# Aggregate Capital Locked per Vendor
inventory_value_per_vendor = df.groupby("VendorName")["UnsoldInventoryValue"].sum().reset_index()

# Sort Vendors with the Highest Locked Capital
inventory_value_per_vendor = inventory_value_per_vendor.sort_values(by="UnsoldInventoryValue", ascending=False)
inventory_value_per_vendor['UnsoldInventoryValue'] = inventory_value_per_vendor['UnsoldInventoryValue'].apply(format_dollars)
inventory_value_per_vendor.head(10)

#### What is the 95% confidence intervals for profit margins of top-performing and low-performing vendors.

In [ ]:
# Define top and low-performing rows using total sales dollars (top 25% and bottom 25%).
top_threshold = df["TotalSalesDollars"].quantile(0.75)
low_threshold = df["TotalSalesDollars"].quantile(0.25)

top_margin_values = df.loc[df["TotalSalesDollars"] >= top_threshold, "ProfitMargin"].dropna()
low_margin_values = df.loc[df["TotalSalesDollars"] <= low_threshold, "ProfitMargin"].dropna()

def confidence_interval(data, confidence=0.95):
    """Return mean and two-sided t confidence interval for a numeric series."""
    values = pd.to_numeric(pd.Series(data), errors="coerce").dropna()
    values = values[np.isfinite(values)]

    if len(values) < 2:
        raise ValueError("At least two valid observations are required for a confidence interval.")

    mean_val = values.mean()
    std_err = values.std(ddof=1) / np.sqrt(len(values))
    t_critical = stats.t.ppf((1 + confidence) / 2, df=len(values) - 1)
    margin_of_error = t_critical * std_err
    return mean_val, mean_val - margin_of_error, mean_val + margin_of_error

top_mean, top_lower, top_upper = confidence_interval(top_margin_values)
low_mean, low_lower, low_upper = confidence_interval(low_margin_values)

print(f"Top-performing group 95% CI: ({top_lower:.2f}, {top_upper:.2f}); mean: {top_mean:.2f}")
print(f"Low-performing group 95% CI: ({low_lower:.2f}, {low_upper:.2f}); mean: {low_mean:.2f}")

plt.figure(figsize=(12, 6))
sns.histplot(top_margin_values, kde=True, color="blue", bins=30, alpha=0.5, label="Top-performing group")
plt.axvline(top_lower, color="blue", linestyle="--", label=f"Top lower: {top_lower:.2f}")
plt.axvline(top_upper, color="blue", linestyle="--", label=f"Top upper: {top_upper:.2f}")
plt.axvline(top_mean, color="blue", linestyle="-", label=f"Top mean: {top_mean:.2f}")

sns.histplot(low_margin_values, kde=True, color="red", bins=30, alpha=0.5, label="Low-performing group")
plt.axvline(low_lower, color="red", linestyle="--", label=f"Low lower: {low_lower:.2f}")
plt.axvline(low_upper, color="red", linestyle="--", label=f"Low upper: {low_upper:.2f}")
plt.axvline(low_mean, color="red", linestyle="-", label=f"Low mean: {low_mean:.2f}")

plt.title("Confidence Interval Comparison: Top vs. Low-Performing Groups")
plt.xlabel("Profit Margin")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**Interpretation guide:** Compare the two confidence intervals and their means after running the updated cell above. If the intervals are well separated, that supports a difference in average profit margin between the selected sales groups; use the Welch t-test below to assess statistical evidence. These are observational group comparisons and do not establish why the margins differ.

#### Is there a significant difference in profit margins between top-performing and low-performing vendors?

Hypothesis:

H₀ (Null Hypothesis): There is no significant difference in the mean profit margins of top-performing and low-performing vendors.

H₁ (Alternative Hypothesis): The mean profit margins of top-performing and low-performing vendors are significantly different.

In [ ]:
top_threshold = df["TotalSalesDollars"].quantile(0.75)
low_threshold = df["TotalSalesDollars"].quantile(0.25)

top_margin_values = df.loc[df["TotalSalesDollars"] >= top_threshold, "ProfitMargin"].dropna()
low_margin_values = df.loc[df["TotalSalesDollars"] <= low_threshold, "ProfitMargin"].dropna()

# Welch's two-sample t-test does not assume equal variances.
t_stat, p_value = ttest_ind(top_margin_values, low_margin_values, equal_var=False, nan_policy="omit")

print(f"Welch t-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6g}")

if p_value < 0.05:
    print("Reject H0: the groups have statistically different mean profit margins (at alpha = 0.05).")
else:
    print("Fail to reject H0: insufficient evidence of a difference in mean profit margins (at alpha = 0.05).")

- A p-value this small suggests that the difference is not just statistically significant but also practically meaningful.

- The two vendor groups operate very differently in terms of profitability.

In [ ]:
# Optional utility: ingest CSV files from a local data/ folder into inventory.db.
# This cell is not required for the analysis if inventory.db or vendor_sales_summary.csv already exists.
from pathlib import Path
import logging
import time
from sqlalchemy import create_engine

DATA_DIR = Path("data")
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=str(LOG_DIR / "ingestion_db.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

engine = create_engine("sqlite:///inventory.db")

def ingest_db(dataframe, table_name, db_engine):
    """Write a DataFrame to a SQLite table."""
    dataframe.to_sql(table_name, con=db_engine, if_exists="replace", index=False)

def load_raw_data():
    """Load CSV files from data/ into matching SQLite tables."""
    if not DATA_DIR.exists():
        print("Optional ingestion skipped: no 'data/' folder was found.")
        return

    csv_files = sorted(DATA_DIR.glob("*.csv"))
    if not csv_files:
        print("Optional ingestion skipped: no CSV files were found in 'data/'.")
        return

    start = time.time()
    for csv_path in csv_files:
        dataframe = pd.read_csv(csv_path)
        table_name = csv_path.stem
        logging.info("Ingesting %s into table %s", csv_path.name, table_name)
        ingest_db(dataframe, table_name, engine)
        print(f"Loaded {csv_path.name} into table {table_name}")

    logging.info("Ingestion complete in %.2f minutes", (time.time() - start) / 60)

# Run only when you explicitly want to ingest CSVs; uncomment the next line if needed.
# load_raw_data()